In [ ]:
#Stock Options pricing-
# To generate stock option prices for a stock for x years, you would need access to an options data source, such as an options exchange or data vendor. You would need to obtain 
# an API key or authentication credentials to access the options data source.

# Once you have access to the options data source, you can use a function to retrieve the option prices for a given stock symbol and date range. Here is an example of a dynamic 
# function that retrieves option prices using the yfinance library:

def get_option_prices(symbol, start_date, end_date):
    start = datetime.strptime(start_date, '%Y-%m-%d') - timedelta(days=30)
    end = datetime.strptime(end_date, '%Y-%m-%d') + timedelta(days=30)
    data = yf.download(symbol, start=start, end=end, group_by='ticker', prepost=True)
    options_data = data['OPTIONS']
    return options_data
# This function takes in the stock symbol, start date, and end date as parameters, and returns a DataFrame containing the option prices for the given stock and date range.

# Note that the prepost parameter is set to True, which allows the function to retrieve option prices outside of regular trading hours. This may not be necessary for all use cases.

# Keep in mind that options prices can be quite volatile and may not always be available for all stocks and date ranges. It's important to carefully consider the quality and 
# availability of the data before using it for any analysis or trading decisions.

#here is a high-level overview of what the function might look like:

#The function would take in the following parameters:

    # ticker: the stock ticker symbol (e.g. "AAPL")
    # start_date: the start date for the data (e.g. "2020-01-01")
    # end_date: the end date for the data (e.g. "2021-12-31")

# The function would use the requests library to make an API call to the CBOE Options API.
# The API response would be in JSON format, so the function would use the json library to parse the response into a Python dictionary.
# The function would then extract the relevant data from the dictionary, such as the option prices, expiration dates, and strike prices.
# The function would return the data in a Pandas DataFrame for further analysis.

# Note that the specifics of the function would depend on the specific API being used, as well as the format of the API response. Additionally, there may be rate limits or other restrictions on API usage that would need to be taken into account.


In [ ]:
#Black-Scholes Fair value comparison 
from scipy.stats import norm

def black_scholes_call(S, K, r, t, sigma):
    d1 = (np.log(S / K) + (r + sigma ** 2 / 2) * t) / (sigma * np.sqrt(t))
    d2 = d1 - sigma * np.sqrt(t)
    return S * norm.cdf(d1) - K * np.exp(-r * t) * norm.cdf(d2)

def option_analysis(ticker, start_date, end_date, strike_price, expiry_date):
    # Download historical stock data
    stock_data = yf.download(ticker, start=start_date, end=end_date)
    stock_data = stock_data.reset_index()

    # Calculate time to expiry in years
    expiry_date = datetime.strptime(expiry_date, "%Y-%m-%d")
    time_to_expiry = (expiry_date - stock_data['Date']).dt.days / 365.0

    # Calculate stock price and risk-free rate
    stock_price = stock_data['Close'].values[-1]
    risk_free_rate = 0.02

    # Calculate annualized volatility of the stock price
    log_returns = np.log(stock_data['Close'] / stock_data['Close'].shift(1))
    volatility = np.sqrt(252) * log_returns.std()

    # Calculate the fair value price of the call option
    fair_value_price = black_scholes_call(stock_price, strike_price, risk_free_rate, time_to_expiry, volatility)

    # Retrieve option data
    option_data = yf.Ticker(f"{ticker}{expiry_date.strftime('%y%m%d')}C{strike_price}").option_chain(expiry_date.strftime("%Y-%m-%d")).calls

    # Get the option price and calculate the percent difference from the fair value price
    option_price = option_data[option_data['inTheMoney'] == False]['lastPrice'].values[0]
    percent_diff = (option_price - fair_value_price) / fair_value_price

    # Return the fair value price, option price, and percent difference
    return fair_value_price, option_price, percent_diff

In [ ]:
#Here's an example script that combines the output of the Black-Scholes pricing model with the Keras model to identify which indicator has high accuracy and is associated with 
# good stock option prices:
from keras.models import load_model
from black_scholes import black_scholes

# Load the Keras model
model = load_model('keras_model.h5')

# Load the accuracy and stock options data
accuracy_data = pd.read_csv('accuracy_features.csv')
options_data = pd.read_csv('stock_options_data.csv')

# Merge the accuracy and stock options data on the date column
data = pd.merge(accuracy_data, options_data, on='Date')

# Calculate the fair value price using the Black-Scholes pricing model
data['Fair Value'] = black_scholes(data['Spot Price'], data['Strike'], data['Risk-Free Rate'], data['Time to Maturity'], data['Volatility'])

# Use the Keras model to predict if the option is a good price
predictions = model.predict(data[['Indicator Accuracy', 'Fair Value']])

# Add the predictions to the dataframe
data['Good Price Prediction'] = predictions

# Print the top 10 rows of the dataframe
print(data.head(10))

This script loads the Keras model, as well as the accuracy and stock options data. It merges the two datasets on the date column and calculates the fair value price using the Black-Scholes pricing model. It then uses the Keras model to predict if the option is a good price, based on the indicator accuracy and the fair value price. Finally, it adds the predictions to the dataframe and prints the top 10 rows.